# UABR-SLM · Notebook 09 — FINAL TEST EVALUATION (the sealed test split is opened once)

GPU **T4** (×1) · ~1 h · No training. **Run this notebook once.** Every rule below was fixed on 12 Nov 2026 before any test
number was seen (PLAN §29). Integrity checks on **validation** run first; the test cells run only if they pass.

**Test set:** in_main PopQA test questions; frozen θ0 labels y (λ from notebook 06) computed in notebook 06 and never used since.

**Models (both seeds 11 / 29 unless E0):** E0 θ_route,preSFT · E1 plain SFT (07) · **A1** SFT-prompt anchoring, **A2** question-pool
anchoring (12) · replay arms (08): **E2** random, **E3** Hybrid-CASR-adapted, **E4** OPR-adapted, **E5** boundary-only (BPR),
**E6** benefit-only, **E7** UABR · comparators needed by the primary family: **interpolation α = 0.5**, **task vector β = 1**
(no training, rebuilt from checkpoints) · critic: **critic** (10, pre-SFT), **critic-SFT** (10), **A2-critic** (12).

| Item | Fixed rule |
|---|---|
| Primary endpoint | routing AUROC vs the frozen θ0 labels on test |
| Primary family | **A2** and **A1** each vs E1, interpolation α = 0.5, task vector β = 1, E2 random, E7 UABR |
| Original hypothesis (reported) | **E7 UABR** vs E1, E2 random, E5 BPR |
| Critic family | **A2-critic** vs critic-SFT on IsRel AUROC (test windows, automatic labels; G2 is validation-only) |
| Test | paired subject-clustered bootstrap, **2,000** resamples, 95 % CI, separately per seed |
| Claim rule | "X preserves the policy better than Y" only if the CI excludes 0 in favour of X in **both** seeds; one seed = "suggestive"; CI excluding 0 against X in both seeds = "worse"; otherwise "no difference detected". No multiplicity correction (the both-seed rule is the guard) |
| Secondary (no claims) | AUPRC, default call rate, agreement with E0, instruction val loss / SFT gain kept, utility at 10 % call budget (θ0 rewards), fresh closed / evidence EM |

**Inputs:** outputs of notebooks **01, 02, 05, 06 (v2), 07, 08, 10, 12**.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.eval import prompts as PR
from uabr.eval.batched import greedy_batch
from uabr.eval.generate import load_model
from uabr.eval.metrics import alias_em
from uabr.retrieval.select import select_non_overlapping
from uabr.routing.metrics import auroc, auprc, clustered_bootstrap_auroc
from uabr.routing.pilot import paired_cluster_bootstrap, top_fraction, outcome
from uabr.routing.router import router_prompt, router_scores
from uabr.critic.selfrag import critique_scores
from uabr.critic.windows import window_items
from uabr.analysis.readout import interpolate, task_vector_add
from uabr.train.sft import sft_val_loss

## 2 · Inputs and the checkpoint list

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/09_test" if ON_KAGGLE else os.path.abspath("outputs/09_test")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4"
SEEDS = [11, 29]; N_BOOT = int(os.environ.get("N_BOOT", 2000)); MAX_NEW, SEQ = 16, 512

def one_input(pattern, env=None):
    if env and os.environ.get(env): return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert len(hits) == 1, f"{pattern}: expected exactly 1 attached file, found {len(hits)}: {hits}"
    return hits[0]
tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20")).set_index("id")
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
THETA0 = one_input("05_mixed/arm_M/model_final.pt", "THETA0")
D06 = os.path.dirname(one_input("06_routing/routing_policy.json", "POLICY")); ROUTER = os.path.join(D06, "router_action.pt")
LAM = float(json.load(open(os.path.join(D06, "routing_policy.json")))["lambda"])
D07 = os.path.dirname(one_input("07_sft_pilot/pilot_report.json", "REPORT07"))
D08 = os.path.dirname(one_input("08_replay/meta.json", "META08")); META08 = json.load(open(os.path.join(D08, "meta.json")))
D10 = os.path.dirname(one_input("10_selfrag/critic.pt", "CRITIC"))
D12 = os.path.dirname(one_input("12_methods/report.json", "REPORT12"))
V = pd.read_parquet(os.path.join(D06, "labels_val.parquet")).reset_index(drop=True)
TE = pd.read_parquet(os.path.join(D06, "labels_test.parquet")).reset_index(drop=True)
assert TE.y.nunique() == 2
WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]
sx = pd.read_parquet(os.path.join(D07, "sft_examples.parquet"))
mk = lambda r: (list(map(int, r.tokens)), [False] * int(r.n_prompt) + [True] * (len(r.tokens) - int(r.n_prompt)))
SFT_VALX = [mk(r) for r in sx[sx.part == "val"].itertuples(index=False)]

def find(d, rel):
    p = os.path.join(d, rel)
    if os.path.exists(p): return p
    hits = sorted(glob.glob(f"/kaggle/input/**/{rel}", recursive=True)); return hits[0] if hits else None
# 08 checkpoints: E2 seed 11 is the beta-tuning run at the chosen beta; paths in meta.json are Kaggle paths of that run
ck08 = {(a, int(s)): p for a, s, p in META08["checkpoints"] if a.startswith("E") and a not in ("E0", "E1")}
def loc08(p):
    rel = p.split("08_replay/")[-1]; return find(D08, rel)
CK = {("E0", "all"): ROUTER}
for s in SEEDS:
    CK[("E1", s)] = os.path.join(D07, f"sft_s{s}", "final.pt")
    CK[("A1", s)] = find(D12, f"A1_sftslot_s{s}/final.pt"); CK[("A2", s)] = find(D12, f"A2_pool_s{s}/final.pt")
    for arm in ["E2", "E3", "E4", "E5", "E6", "E7"]: CK[(arm, s)] = loc08(ck08[(arm, s)])
CRIT = {("critic", "all"): os.path.join(D10, "critic.pt")}
for s in SEEDS:
    CRIT[("critic_SFT", s)] = os.path.join(D10, f"sft_s{s}", "final.pt"); CRIT[("A2c", s)] = find(D12, f"A2_critic_s{s}/final.pt")
missing = [k for k, p in {**CK, **CRIT}.items() if not p or not os.path.exists(p)]
assert not missing, f"missing checkpoints: {missing} - attach the outputs of 07, 08, 10, 12"
sd = lambda p: torch.load(p, map_location="cpu", weights_only=False)["model"]
SD_R, SD_T0 = sd(ROUTER), sd(THETA0); cfg0 = load_model(ROUTER, "cpu").cfg.to_dict()
for s in SEEDS:
    SD_S = sd(CK[("E1", s)])
    for name, st in [("interp0.5", interpolate(SD_R, SD_S, 0.5)), ("taskvec1", task_vector_add(SD_S, SD_R, SD_T0, 1.0))]:
        p = f"{OUT}/ref_{name}_s{s}.pt"; torch.save({"model": st, "model_cfg": cfg0}, p); CK[(name, s)] = p
print(f"test questions {len(TE):,} (CALL labels {TE.y.mean():.1%}) | routing models {len(CK)} | critic models {len(CRIT)}")

## 3 · Integrity checks on VALIDATION (must pass before any test cell runs)

In [ ]:
RP_V = [router_prompt(tok, q, "action") for q in V.question]; yV = V.y.astype(int).values
VAL = {}
for key in [("E0", "all"), ("E1", 11), ("E1", 29), ("A2", 11), ("E7", 11), ("interp0.5", 11)]:
    m = load_model(CK[key], DEV); VAL[key] = auroc(yV, router_scores(m, RP_V, "action", DEV)[0]); del m
p07 = pd.read_csv(os.path.join(D07, "pilot_table.csv")); p07 = p07[p07.frac == 1.0]; ref07 = {int(r.seed): r.auroc for r in p07.itertuples()}
r08 = pd.read_csv(os.path.join(D08, "results_val.csv")); e7 = r08[(r08.arm == "E7") & (r08.seed.astype(str) == "11")].auroc.item()
r12 = pd.read_csv(os.path.join(D12, "results_routing.csv"))
a2 = r12[(r12.method == "A2") & (r12.seed.astype(str) == "11")].auroc_full_val.item(); i05 = r12[(r12.method == "interp0.5") & (r12.seed.astype(str) == "11")].auroc_full_val.item()
INTEG = {"E0 = 06 router": abs(VAL[("E0", "all")] - auroc(yV, V.router_score)) < 1e-3, "E1 = 07": all(abs(VAL[("E1", s)] - ref07[s]) < 1e-3 for s in SEEDS),
         "E7 = 08": abs(VAL[("E7", 11)] - e7) < 1e-3, "A2 = 12": abs(VAL[("A2", 11)] - a2) < 1e-3, "interp0.5 = 12": abs(VAL[("interp0.5", 11)] - i05) < 2e-3}
print({k: (round(v, 4)) for k, v in VAL.items()}); print(INTEG)
assert all(INTEG.values()), f"integrity check failed - wrong checkpoint attached? {INTEG}"
print("validation integrity: PASS -> opening the test split")

## 4 · TEST: routing models

In [ ]:
RP_T = [router_prompt(tok, q, "action") for q in TE.question]; yT, gT = TE.y.astype(int).values, TE.group_id.values
CPT = [Q.closed_prompt_ids(tok, q) for q in TE.question]
EPT = []
for r in TE.itertuples(index=False):
    sel, _ = select_non_overlapping(list(top.loc[r.id].doc_ids), WART, WSTART, k=3, window_words=60)
    EPT.append(Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)[0])
def em(m, prompts):
    g = greedy_batch(m, prompts, DEV, MAX_NEW, banned=(T.CALL_ID,), stop_ids=stop_ids)
    return np.array([alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(g, TE.gold_aliases)])
S, ROWS = {}, []
for key, path in sorted(CK.items(), key=lambda kv: str(kv[0])):
    m = load_model(path, DEV); s = router_scores(m, RP_T, "action", DEV)[0]; S[key] = s
    ci = clustered_bootstrap_auroc(yT, s, gT, n_boot=N_BOOT); fa, fc = em(m, CPT), em(m, EPT)
    call10 = top_fraction(s, 0.10)
    ROWS.append({"model": key[0], "seed": key[1], "auroc": ci["auroc"], "lo": ci["lo"], "hi": ci["hi"], "auprc": auprc(yT, s),
                 "default_call": float((s > 0).mean()), "instr_val_loss": sft_val_loss(m, SFT_VALX, DEV),
                 "U10_theta0": outcome(call10, TE.em_A, TE.em_C)["accuracy"] - LAM * 0.10,
                 "U10_fresh": outcome(call10, fa, fc)["accuracy"] - LAM * 0.10, "closed_em": fa.mean(), "evidence_em": fc.mean()})
    del m
    if DEV == "cuda": torch.cuda.empty_cache()
    r = ROWS[-1]; stage(f"{key[0]} s{key[1]}: test AUROC {r['auroc']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}] | call {r['default_call']:.1%} | closed {r['closed_em']:.1%} | evidence {r['evidence_em']:.1%}")
D0 = S[("E0", "all")] > 0
RES = pd.DataFrame(ROWS); RES["agree_E0"] = [float(((S[(m, s)] > 0) == D0).mean()) for m, s in zip(RES.model, RES.seed)]
L0 = RES.loc[RES.model == "E0", "instr_val_loss"].item()
RES["sft_gain_kept"] = [0.0 if s == "all" else (L0 - l) / (L0 - RES[(RES.model == "E1") & (RES.seed == s)].instr_val_loss.item()) for l, s in zip(RES.instr_val_loss, RES.seed)]
RES.to_csv(f"{OUT}/test_routing.csv", index=False); pd.set_option("display.width", 250); display(RES.round(4))

## 5 · TEST: critic models (IsRel on test windows, automatic labels)

In [ ]:
TW = window_items(TE, top, tok, WTEXT, WART, WSTART, 5, SEQ - MAX_NEW)
print(f"test windows {len(TW):,} (relevant {TW.auto_rel.mean():.1%}; dropped {TW.attrs['n_dropped']})")
CS, CROWS = {}, []
for key, path in CRIT.items():
    m = load_model(path, DEV); CS[key] = critique_scores(m, TW.prompt.tolist(), DEV); rs = router_scores(m, RP_T, "action", DEV)[0]
    ci = clustered_bootstrap_auroc(TW.auto_rel.values, CS[key], TW.group_id.values, n_boot=N_BOOT)
    CROWS.append({"model": key[0], "seed": key[1], "isrel_auroc": ci["auroc"], "lo": ci["lo"], "hi": ci["hi"], "route_auroc": auroc(yT, rs),
                  "instr_val_loss": sft_val_loss(m, SFT_VALX, DEV)}); del m
CRES = pd.DataFrame(CROWS); CRES.to_csv(f"{OUT}/test_critic.csv", index=False); display(CRES.round(4))

## 6 · TEST: pre-registered comparisons and the claim rule

In [ ]:
def cmp_route(a, b, seed):
    f = lambda idx, w: auroc(yT[idx], (S[(a, seed)] if w == "a" else S[(b, seed)])[idx])
    return paired_cluster_bootstrap(f, gT, n_boot=N_BOOT, seed=seed)
FAMILIES = {"primary": [(a, b) for a in ["A2", "A1"] for b in ["E1", "interp0.5", "taskvec1", "E2", "E7"]],
            "original_hypothesis": [("E7", b) for b in ["E1", "E2", "E5"]],
            "descriptive_vs_E1": [(a, "E1") for a in ["E2", "E3", "E4", "E5", "E6"]]}
rows = []
for fam, pairs in FAMILIES.items():
    for a, b in pairs:
        r = {s: cmp_route(a, b, s) for s in SEEDS}
        rows.append({"family": fam, "comparison": f"{a} - {b}", "metric": "routing AUROC", **{f"s{s}_{k}": r[s][k] for s in SEEDS for k in ("diff", "ci_lo", "ci_hi")}})
rc = {}
for s in SEEDS:
    f = lambda idx, w, s=s: auroc(TW.auto_rel.values[idx], (CS[("A2c", s)] if w == "a" else CS[("critic_SFT", s)])[idx])
    rc[s] = paired_cluster_bootstrap(f, TW.group_id.values, n_boot=N_BOOT, seed=s)
rows.append({"family": "critic", "comparison": "A2c - critic_SFT", "metric": "IsRel AUROC", **{f"s{s}_{k}": rc[s][k] for s in SEEDS for k in ("diff", "ci_lo", "ci_hi")}})
CMP = pd.DataFrame(rows)
def verdict(r):
    better = [r[f"s{s}_ci_lo"] > 0 for s in SEEDS]; worse = [r[f"s{s}_ci_hi"] < 0 for s in SEEDS]
    if all(better): return "BETTER (both seeds)"
    if all(worse): return "WORSE (both seeds)"
    if any(better): return "suggestive (one seed)"
    return "no difference detected"
CMP["verdict"] = CMP.apply(verdict, axis=1); CMP.to_csv(f"{OUT}/test_comparisons.csv", index=False)
display(CMP.round(4))
for r in CMP.itertuples():
    print(f"[{r.family}] {r.comparison} ({r.metric}): s11 {r.s11_diff:+.3f} [{r.s11_ci_lo:+.3f}, {r.s11_ci_hi:+.3f}] | "
          f"s29 {r.s29_diff:+.3f} [{r.s29_ci_lo:+.3f}, {r.s29_ci_hi:+.3f}] -> {r.verdict}")

## 7 · Save + report

In [ ]:
checks = [{"name": "validation_integrity", "passed": all(INTEG.values()), "detail": str(INTEG), "severity": "fail"},
          {"name": "test_labels_two_classes", "passed": TE.y.nunique() == 2, "detail": f"CALL labels {TE.y.mean():.1%}", "severity": "fail"}]

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"notebook": "09_test", "status": status, "n_test": len(TE), "n_boot": N_BOOT, "verdicts": CMP[["family", "comparison", "metric", "verdict"]].to_dict("records"),
           "checks": checks, "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}}, open(f"{OUT}/test_report.json", "w"), indent=1, default=float)
print(f"\nNOTEBOOK 09 (TEST): {status}")